# MedGemma-4B hallucination evaluation — Kaggle T4

Self-contained evaluation run for the clinical-VLM trustworthiness project.
Generated by `model/scripts/build_eval_notebook.py` — do not hand-edit; edit
`model/src/predict.py` / `model/config.json` in the repo and regenerate.

## Before running

1. **Settings → Accelerator → GPU T4 x2** (or T4 x1). Only GPU 0 is used.
2. **Settings → Internet → On** (needed to reach Hugging Face).
3. Add two Kaggle secrets:
   - **`HF_TOKEN`** — read access, used for every download.
   - **`HF_WRITE_TOKEN`** — write access to `Arjunsatheesh/clinical-vlm-medgemma-lora`, used only by the
     final upload cell. Its presence is checked early so a missing secret
     fails in seconds rather than after the full run.
4. Accept the `google/medgemma-4b-it` licence on Hugging Face with the same
   account — it is a gated model.

## Numeric precision

`float16` is never used. On a T4 this model produces **NaN logits and empty
output** in fp16 (measured by the project owner). `compute_dtype()` therefore
returns `bfloat16` only where bf16 is native (compute capability >= 8.0, e.g.
an RTX 3050 at 8.6) and **`float32` everywhere else**, including the T4 at 7.5.
The adapter was trained with float32 compute, so that is the consistent choice.
The same dtype is applied to the 4-bit compute and to every module
bitsandbytes leaves unquantised.

After the first generation of each configuration the harness aborts the run if
the logits are not finite or the decoded text is empty, so a bad numeric setup
cannot quietly produce a file full of blank answers.

Then **Save Version → Save & Run All (Commit)**.

## What this runs

Three configurations, each over `val.json` first and then `test.json`:

| config | adapter | prompt |
|---|---|---|
| `baseline` | none | plain clinical VQA |
| `adapter_train` | LoRA | the exact template the adapter was trained on |
| `adapter_strict` | LoRA | strict visible-evidence + abstention prompt |

`adapter_train` vs `adapter_strict` isolates the train/inference prompt
mismatch: `train_info.json` shows the adapter was fine-tuned on
`"Answer the clinical question about this image briefly. {q}"`, which is not
the strict prompt, so comparing them separates "the adapter helps" from "the
strict prompt helps".

Output: one JSONL row per example at `results/<split>_<config>.jsonl`,
appended and fsynced per row, then uploaded to the HF repo.

## What this deliberately does NOT do

- **No abstention threshold is applied or chosen.** `abstention_threshold` is
  forced to `0.0`, and every row stores `raw_text`, `confidence` and
  `gold_answer` untouched, so thresholds can be swept offline afterwards.
- **No metrics.** No accuracy, no hallucination rate, no calibration. The last
  cell prints row counts only.
- No training. Inference only.

Expect roughly 3 x (220 + 337) = 1671 generations plus 3 model loads.

## 1. Environment

In [ ]:
import subprocess, sys

# Kaggle ships torch; these are the extras predict.py needs. Pinned loosely to
# match what the project uses locally (see model/requirements-model.txt).
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.50", "accelerate>=0.34", "peft>=0.13",
    "bitsandbytes>=0.43.3",
    # >=0.34 is required for the `hf` CLI used below; Kaggle's preinstalled
    # version can be older, and a looser pin would leave it in place.
    "huggingface_hub>=0.34",
    "pillow>=10.0",
], check=True)

import shutil
if shutil.which("hf") is None:
    raise SystemExit(
        "The `hf` CLI is missing even after installing huggingface_hub>=0.34. "
        "Restart the session so the new console scripts are on PATH, or replace "
        "the `!hf download` calls with huggingface_hub.snapshot_download."
    )

import torch, transformers, peft, bitsandbytes, accelerate
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("accelerate  :", accelerate.__version__)
print("CUDA        :", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU         : {p.name}  {p.total_memory/1024**3:.1f} GB  "
          f"compute {p.major}.{p.minor}")
    print("torch says bf16 ok:", torch.cuda.is_bf16_supported(),
          "(True on a T4 only because it emulates bf16)")
    native_bf16 = p.major >= 8
    print("native bf16       :", native_bf16)
    print("dtype that will be used:",
          "bfloat16" if native_bf16 else "float32",
          "- float16 is never selected (NaN logits on T4)")
else:
    raise SystemExit("No GPU. Set Accelerator to GPU T4 before running.")

## 2. Write out `predict.py` and `config.json`

The repo layout is recreated so that `predict.py`'s `_CONFIG_PATH` (`parents[1] / "config.json"`) still resolves: the file must sit at `model/src/predict.py` with the config at `model/config.json`.

In [ ]:
import os
from pathlib import Path

for d in ("model/src", "model/notebooks", "results"):
    os.makedirs(d, exist_ok=True)
# Namespace packages would work, but the repo has these and they cost nothing.
Path("model/__init__.py").touch()
Path("model/src/__init__.py").touch()
print(sorted(os.listdir(".")))

In [ ]:
%%writefile model/src/predict.py
"""Public interface of the model module.

Real MedGemma-4B implementation. The signature and the `Prediction` fields are
unchanged from the stub version (see model/docs/INTERFACE.md); only the body of
`predict()` is real now.

The model is loaded once, lazily, on the first real call and cached at module
level. Loading settings are the ones measured to fit this hardware in
model/docs/FIT_TEST.md: 4-bit nf4, bf16 compute, device_map={"": 0}.

If CUDA is unavailable the module falls back to the old stub answers with
is_stub=True, so the Streamlit app and the evaluation harness still import and
run on a machine without a GPU. torch and transformers are imported lazily for
the same reason - importing this module does not require them.
"""
from __future__ import annotations

import json
import math
import os
import re
import threading
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Literal, Optional, Union

from PIL import Image

Mode = Literal["baseline", "mitigated"]

_MODEL_DIR = Path(__file__).resolve().parents[1]
_CONFIG_PATH = _MODEL_DIR / "config.json"

_DEFAULTS = {
    "model_id": "google/medgemma-4b-it",
    "abstention_threshold": 0.8,
    "max_new_tokens": 48,
    "min_yes_no_mass": 0.10,
    "default_mitigated_prompt_style": "train",
    "abstain_answer": "cannot determine from this image",
}

# Hard ceiling regardless of config: only ~0.18 GB of VRAM is free after load.
_MAX_NEW_TOKENS_CAP = 48

# Questions that start like this are treated as closed (yes/no), which switches
# confidence to the yes-vs-no probability mass on the first generated token.
_CLOSED_START = re.compile(
    r"^\s*(is|are|was|were|does|do|did|can|could|has|have|had|will|would|should|any)\b",
    re.IGNORECASE,
)

_YES_WORDS = ["yes", "Yes", "YES", " yes", " Yes"]
_NO_WORDS = ["no", "No", "NO", " no", " No"]


@dataclass
class Prediction:
    answer: str            # final answer text shown to the user
    confidence: float      # 0.0 - 1.0, from generated-token probabilities
    abstained: bool        # True if the model declined to answer
    raw_text: str          # unprocessed model generation
    mode: str              # "baseline" or "mitigated"
    is_stub: bool = True   # False once the real model is wired in

    def to_dict(self) -> dict:
        return asdict(self)


def load_config() -> dict:
    """Read model/config.json over the defaults. Missing or broken file -> defaults."""
    config = dict(_DEFAULTS)
    try:
        with open(_CONFIG_PATH, encoding="utf-8") as fh:
            on_disk = json.load(fh)
    except (FileNotFoundError, json.JSONDecodeError):
        return config
    for key in _DEFAULTS:
        if key in on_disk:
            config[key] = on_disk[key]
    return config


# --------------------------------------------------------------------------
# Model cache
# --------------------------------------------------------------------------

_lock = threading.Lock()
_cache: dict = {"key": None, "model": None, "processor": None}

#: Diagnostics for the most recent predict() generation. Callers use this to
#: detect a broken numeric setup (e.g. fp16 NaN logits on a T4) which cannot be
#: expressed through Prediction's fixed fields. Overwritten on every call.
LAST_GENERATION: dict = {
    "scores_finite": None,     # True/False: every generated step's logits finite
    "non_finite_steps": None,  # how many steps contained NaN/Inf
    "new_tokens": None,        # number of tokens generated
    "text_empty": None,        # True if the decoded text is blank
    "dtype": None,             # compute dtype actually used
    "device_capability": None, # e.g. "8.6"
}


def cuda_available() -> bool:
    """True only if torch is importable and reports a usable CUDA device.

    Importing torch can fail with more than ImportError. On this machine
    Windows Smart App Control blocked torch's unsigned torch_python.dll, which
    surfaces as OSError [WinError 4551]. Any import failure means "no usable
    GPU stack", so callers get the stub rather than a crash.
    """
    try:
        import torch
    except Exception:  # noqa: BLE001 - ImportError, OSError (blocked DLL), etc.
        return False
    try:
        return torch.cuda.is_available()
    except Exception:  # noqa: BLE001 - a broken driver must not raise here
        return False


#: GPUs from compute capability 8.0 (Ampere) up have bf16 tensor cores.
_NATIVE_BF16_MAJOR = 8


def compute_dtype(torch):
    """bfloat16 only on GPUs with NATIVE bf16, otherwise float32. Never float16.

    float16 is deliberately excluded: on a Kaggle T4 this model produces NaN
    logits and empty output in fp16 (measured by the project owner). bf16
    emulated and float32 both work, and the adapter was trained with float32
    compute, so float32 is the safe fallback.

    The test is compute capability >= 8.0, not torch.cuda.is_bf16_supported():
    recent torch reports True on a T4 because it will emulate bf16, which is
    not what we want to select on. An RTX 3050 is 8.6 and takes the bf16 path;
    a T4 is 7.5 and takes float32.
    """
    try:
        if torch.cuda.is_available():
            major, _minor = torch.cuda.get_device_capability(0)
            if major >= _NATIVE_BF16_MAJOR:
                return torch.bfloat16
    except Exception:  # noqa: BLE001 - never let a probe failure pick fp16
        pass
    return torch.float32


def _build_model(model_id: str, adapter_path: Optional[str]):
    """Load the 4-bit model and processor. Settings verified in FIT_TEST.md."""
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

    dtype = compute_dtype(torch)

    quantisation = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        # Same dtype for the 4-bit compute and, via dtype= below, for every
        # module bitsandbytes leaves unquantised (embeddings, norms, lm_head).
        # Mixing fp32 weights with an fp16 compute dtype is what produced the
        # NaN logits on a T4, so the two must not diverge.
        bnb_4bit_compute_dtype=dtype,
    )

    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        quantization_config=quantisation,
        # device_map="auto" is NOT usable on a 4 GB card: accelerate reserves
        # activation headroom, decides the model will not fit and refuses to
        # place it. Forcing GPU 0 is the configuration measured to work, and is
        # also correct on a single-GPU T4.
        device_map={"": 0},
        dtype=dtype,
    )

    if adapter_path:
        from peft import PeftModel

        model = PeftModel.from_pretrained(model, adapter_path)

    model.eval()
    return model, processor


def get_model(adapter_path: Optional[str] = None, model_id: Optional[str] = None):
    """Return the cached (model, processor), loading on first use.

    Reloads if a different adapter_path is requested than the cached one.
    """
    model_id = model_id or load_config()["model_id"]
    key = (model_id, adapter_path or "")
    with _lock:
        if _cache["key"] != key or _cache["model"] is None:
            _cache["model"], _cache["processor"] = _build_model(model_id, adapter_path)
            _cache["key"] = key
        return _cache["model"], _cache["processor"]


# --------------------------------------------------------------------------
# Prompts
# --------------------------------------------------------------------------

_BASELINE_PROMPT = (
    "You are assisting with reading a medical image. "
    "Answer the following question about the image.\n\nQuestion: {question}"
)

# The "strict" prompt. Selectable, but NOT the mitigated default: evaluated at
# threshold 0.8 it abstains on ~79% of val questions (70% of test), which makes
# the system useless in practice. Kept for ablation and error analysis only.
_MITIGATED_PROMPT = (
    "You are assisting with reading a medical image. Answer the question using "
    "ONLY what is visibly present in this image.\n"
    "Rules:\n"
    "- Do not mention any finding, organ or device you cannot actually see.\n"
    "- Do not guess, and do not infer from what is usually present in such images.\n"
    '- If the thing asked about is not clearly visible, or you are unsure, reply '
    'exactly: "{abstain}"\n\nQuestion: {question}'
)

# Copied verbatim from the adapter's train_info.json "prompt_template". The {q}
# placeholder is substituted literally rather than reformatted, so this is
# provably the string the adapter was fine-tuned on. Use prompt_style="train"
# to remove the train/inference prompt mismatch as a confound.
_TRAIN_PROMPT_TEMPLATE = "Answer the clinical question about this image briefly. {q}"

PROMPT_STYLES = ("plain", "strict", "train")


def default_prompt_style(mode: str) -> str:
    """Which prompt a mode uses when prompt_style is not given.

    baseline  -> "plain" (and no adapter).
    mitigated -> config.json's default_mitigated_prompt_style, "train" by
                 default: the exact template the adapter was fine-tuned on.
                 "strict" remains selectable per call but is deliberately not
                 the default - see the note on _MITIGATED_PROMPT.
    """
    if mode == "baseline":
        return "plain"
    style = load_config()["default_mitigated_prompt_style"]
    return style if style in PROMPT_STYLES else "train"


def build_prompt(question: str, mode: str, abstain_answer: str,
                 prompt_style: Optional[str] = None) -> str:
    """Render the user-turn text for one question."""
    style = prompt_style or default_prompt_style(mode)
    if style not in PROMPT_STYLES:
        raise ValueError(f"prompt_style must be one of {PROMPT_STYLES}, got {style!r}")
    if style == "train":
        return _TRAIN_PROMPT_TEMPLATE.replace("{q}", question)
    if style == "strict":
        return _MITIGATED_PROMPT.format(question=question, abstain=abstain_answer)
    return _BASELINE_PROMPT.format(question=question)


def _build_messages(image: Image.Image, question: str, mode: str, abstain_answer: str,
                    prompt_style: Optional[str] = None):
    text = build_prompt(question, mode, abstain_answer, prompt_style)
    return [{
        "role": "user",
        "content": [{"type": "image", "image": image}, {"type": "text", "text": text}],
    }]


# --------------------------------------------------------------------------
# Confidence
# --------------------------------------------------------------------------

def is_closed_question(question: str) -> bool:
    """Heuristic yes/no detector. predict() is not given the question_type."""
    return bool(_CLOSED_START.match(question)) or "yes or no" in question.lower()


def _first_token_ids(tokenizer, words: list[str]) -> list[int]:
    """First token id of each spelling, deduplicated."""
    ids = []
    for word in words:
        encoded = tokenizer.encode(word, add_special_tokens=False)
        if encoded:
            ids.append(encoded[0])
    return sorted(set(ids))


def _mean_token_probability(scores, sequences, prompt_len: int) -> Optional[float]:
    """Mean probability of the tokens the model actually generated."""
    import torch

    probs = []
    for step, step_logits in enumerate(scores):
        index = prompt_len + step
        if index >= sequences.shape[1]:
            break
        distribution = torch.softmax(step_logits[0].float(), dim=-1)
        probs.append(distribution[sequences[0, index]].item())
    if not probs:
        return None
    return sum(probs) / len(probs)


def _yes_no_confidence(first_step_logits, tokenizer,
                       min_mass: float = 0.10) -> Optional[float]:
    """Probability mass of yes vs no on the first generated token.

    Returns the winning side's share, so 0.5 means a coin flip and 1.0 means all
    of the yes/no mass sits on one side.

    Returns None when yes and no together hold less than `min_mass` of the
    distribution, because the ratio is scale-free and would otherwise report a
    confident-looking number off almost no probability. Measured on this model:
    a real "Yes" carries ~0.97 of the first-token mass, while a hedged prose
    answer carries ~0.03 and a refusal ~0.0005 - and the unguarded ratio turned
    those last two into 0.89 and 0.97. In mitigated mode that is the dangerous
    direction, since inflated confidence suppresses abstention. The caller falls
    back to the mean generated-token probability instead.
    """
    import torch

    distribution = torch.softmax(first_step_logits[0].float(), dim=-1)
    p_yes = distribution[_first_token_ids(tokenizer, _YES_WORDS)].sum().item()
    p_no = distribution[_first_token_ids(tokenizer, _NO_WORDS)].sum().item()
    total = p_yes + p_no
    if total < max(min_mass, 1e-6):
        return None
    return max(p_yes, p_no) / total


def _clean(text: str) -> str:
    return " ".join(text.split()).strip()


# --------------------------------------------------------------------------
# Stub fallback (no CUDA)
# --------------------------------------------------------------------------

def _stub_prediction(mode: str, abstain_answer: str) -> Prediction:
    if mode == "mitigated":
        return Prediction(abstain_answer, 0.5, True, f"[stub] {abstain_answer}", mode)
    return Prediction("yes", 0.5, False, "[stub] yes", mode)


# --------------------------------------------------------------------------
# Public entry point
# --------------------------------------------------------------------------

def predict(
    image: Union[str, os.PathLike, Image.Image],
    question: str,
    mode: Mode = "baseline",
    adapter_path: Optional[str] = None,
    prompt_style: Optional[str] = None,
) -> Prediction:
    """Answer one clinical question about one image.

    mode="baseline"  - plain prompt, no adapter, never abstains.
    mode="mitigated" - abstains when the confidence falls below config.json's
                       abstention_threshold (0.8 as evaluated). Uses the
                       training template by default.
    adapter_path     - optional LoRA adapter directory, loaded with peft. The
                       evaluated system passes
                       model/checkpoints/medgemma_vqarad_lora.
    prompt_style     - override the prompt: "plain", "strict" or "train".
                       Default None keeps each mode's own prompt: baseline ->
                       "plain", mitigated -> "train" (the exact template the
                       adapter was fine-tuned on). "strict" is selectable but
                       is not the default; it abstains on ~79% of val
                       questions.
    """
    if mode not in ("baseline", "mitigated"):
        raise ValueError("mode must be 'baseline' or 'mitigated'")
    if isinstance(image, (str, os.PathLike)):
        image = Image.open(image)
    image = image.convert("RGB")

    config = load_config()
    abstain_answer = config["abstain_answer"]

    if not cuda_available():
        return _stub_prediction(mode, abstain_answer)

    import torch

    model, processor = get_model(adapter_path=adapter_path, model_id=config["model_id"])
    messages = _build_messages(image, question, mode, abstain_answer, prompt_style)
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]

    max_new_tokens = min(int(config["max_new_tokens"]), _MAX_NEW_TOKENS_CAP)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,              # greedy
            return_dict_in_generate=True,
            output_scores=True,
        )

    sequences = generated.sequences
    scores = generated.scores or ()
    raw_text = processor.decode(sequences[0][prompt_len:], skip_special_tokens=True)
    answer = _clean(raw_text)

    # Record whether the numerics held up. A broken dtype choice shows up here
    # as non-finite logits and/or empty text, which no Prediction field can
    # express - callers check LAST_GENERATION and abort.
    non_finite_steps = sum(
        0 if bool(torch.isfinite(step_logits).all()) else 1 for step_logits in scores
    )
    LAST_GENERATION.update({
        "scores_finite": non_finite_steps == 0 and bool(scores),
        "non_finite_steps": non_finite_steps,
        "new_tokens": int(sequences.shape[1] - prompt_len),
        "text_empty": not raw_text.strip(),
        "dtype": str(compute_dtype(torch)),
        "device_capability": "%d.%d" % torch.cuda.get_device_capability(0),
    })

    confidence = None
    if scores:
        if is_closed_question(question):
            confidence = _yes_no_confidence(
                scores[0], processor.tokenizer,
                min_mass=float(config["min_yes_no_mass"]),
            )
        if confidence is None:
            confidence = _mean_token_probability(scores, sequences, prompt_len)
    # NaN must be caught before clamping: min()/max() compare False against NaN
    # and would quietly return 1.0, turning broken logits into max confidence.
    if confidence is None or not math.isfinite(float(confidence)):
        confidence = 0.0
    confidence = max(0.0, min(1.0, float(confidence)))

    abstained = False
    if mode == "mitigated":
        said_so = abstain_answer.lower() in answer.lower()
        if said_so or confidence < float(config["abstention_threshold"]):
            abstained = True
            answer = abstain_answer

    return Prediction(
        answer=answer,
        confidence=confidence,
        abstained=abstained,
        raw_text=raw_text,
        mode=mode,
        is_stub=False,
    )

In [ ]:
%%writefile model/config.json
{
  "_comment": "Runtime settings for model/src/predict.py. Tunable without code changes.",

  "model_id": "google/medgemma-4b-it",

  "abstention_threshold": 0.8,
  "_abstention_threshold": "mitigated mode abstains when confidence falls below this. 0.0 disables abstention, 1.0 abstains always. 0.8 is the evaluated operating point: on the test split it abstains on 48.4% of questions and cuts committed error from 47.8% to 10.1% (see model/docs/MODEL.md).",

  "default_mitigated_prompt_style": "train",
  "_default_mitigated_prompt_style": "which prompt mitigated mode uses when the caller passes no prompt_style. 'train' is the exact template the adapter was fine-tuned on. 'strict' is still selectable per call but must NOT be the default: it abstains on ~79% of val questions, which makes the system unusable.",

  "max_new_tokens": 48,
  "_max_new_tokens": "hard ceiling is 48; only ~0.18 GB of VRAM is free after the 4-bit model loads (see model/docs/FIT_TEST.md).",

  "min_yes_no_mass": 0.10,
  "_min_yes_no_mass": "the yes-vs-no ratio is only trusted when yes and no together hold at least this much of the first-token distribution. Measured: a real yes/no answer carries ~0.97, while a hedged prose answer carries ~0.03 and a refusal ~0.0005, where the ratio would otherwise invent high confidence. Below this floor, confidence falls back to the mean generated-token probability. Set to 0.0 for the unguarded ratio.",

  "abstain_answer": "cannot determine from this image"
}

### Disable post-hoc thresholding

`abstention_threshold` is set to `0.0` so `predict()` never replaces an answer on confidence grounds. Thresholds are a downstream analysis choice and are deliberately not made here. The model can still emit the abstain phrase by itself under the strict prompt; that is model behaviour, and rows record it separately so it is never confused with a threshold decision.

In [ ]:
import json

cfg = json.load(open("model/config.json", encoding="utf-8"))
cfg["abstention_threshold"] = 0.0
cfg["_eval_note"] = ("threshold forced to 0.0 for the evaluation run; sweep "
                     "thresholds offline from confidence + raw_text")
with open("model/config.json", "w", encoding="utf-8") as fh:
    json.dump(cfg, fh, indent=2)

import sys
sys.path.insert(0, ".")
from model.src.predict import (                      # noqa: E402
    load_config, predict, cuda_available, compute_dtype,
    build_prompt, PROMPT_STYLES,
)

live = load_config()
print("model_id            :", live["model_id"])
print("abstention_threshold:", live["abstention_threshold"], "(0.0 = not applied)")
print("min_yes_no_mass     :", live["min_yes_no_mass"])
print("max_new_tokens      :", live["max_new_tokens"])
print("prompt styles       :", PROMPT_STYLES)
print("compute dtype       :", compute_dtype(torch))
assert live["abstention_threshold"] == 0.0

## 3. Hugging Face login and download

Two Kaggle secrets, matching the training notebook:

- **`HF_TOKEN`** — read access, used for all downloads (base model, adapter, data, and recovering previous results).
- **`HF_WRITE_TOKEN`** — write access, used *only* by the upload cell at the end.

Neither is printed, logged or written to disk. The write token is read lazily in the upload cell rather than being used to log in, so the whole run is read-only until that last step.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

HF_REPO = "Arjunsatheesh/clinical-vlm-medgemma-lora"
BASE_MODEL = "google/medgemma-4b-it"

_secrets = UserSecretsClient()
HF_READ_TOKEN = _secrets.get_secret("HF_TOKEN")

login(token=HF_READ_TOKEN, add_to_git_credential=False)
print("logged in for downloads as:", whoami(token=HF_READ_TOKEN)["name"])

# Fail now, not 90 minutes from now, if the write secret is missing.
try:
    _secrets.get_secret("HF_WRITE_TOKEN")
    print("HF_WRITE_TOKEN secret: present (used only by the upload cell)")
except Exception as exc:
    raise SystemExit(
        "HF_WRITE_TOKEN secret is missing. Add it under Add-ons -> Secrets "
        "before running, or the results upload at the end will fail after the "
        f"whole evaluation has already run. ({type(exc).__name__})"
    )

In [ ]:
# Adapter and data land side by side under hf_repo/, which keeps the JSON files'
# image_path values ("images/<id>.jpg") resolvable relative to hf_repo/data.
!hf download Arjunsatheesh/clinical-vlm-medgemma-lora --include "adapter/*" --local-dir hf_repo
!hf download Arjunsatheesh/clinical-vlm-medgemma-lora --include "data/*"    --local-dir hf_repo

ADAPTER   = Path("hf_repo/adapter")
DATA_ROOT = Path("hf_repo/data")

print("adapter files:", sorted(p.name for p in ADAPTER.iterdir()))
print("images       :", len(list((DATA_ROOT / "images").glob("*.jpg"))))
print("splits       :", sorted(p.name for p in DATA_ROOT.glob("*.json")))
print()
print("train_info.json:")
print(json.dumps(json.load(open(ADAPTER / "train_info.json", encoding="utf-8")), indent=2))

### Check every `image_path` resolves before spending GPU time

In [ ]:
from PIL import Image

SPLITS = {}
for name in ("val", "test"):
    rows = json.load(open(DATA_ROOT / f"{name}.json", encoding="utf-8"))
    missing = [r["image_path"] for r in rows if not (DATA_ROOT / r["image_path"]).exists()]
    SPLITS[name] = rows
    print(f"{name:<5} rows={len(rows):<4} unique images="
          f"{len({r['image_path'] for r in rows}):<4} missing={len(missing)}")
    if missing:
        raise SystemExit(f"{len(missing)} image_path values do not resolve, e.g. {missing[:3]}")

# Decode every distinct image once, so a corrupt file fails now and not mid-run.
bad = []
for path in sorted({r["image_path"] for rows in SPLITS.values() for r in rows}):
    try:
        with Image.open(DATA_ROOT / path) as im:
            im.verify()
    except Exception as exc:
        bad.append((path, repr(exc)))
print("images that fail to decode:", len(bad), bad[:3])
assert not bad
print("\nall image paths resolve and decode")

## 4. Resume from any previous run

A Kaggle commit starts with a clean filesystem, so previously finished rows are pulled back from the HF repo. That makes a re-run resume instead of recomputing, whether it was interrupted mid-cell or in an earlier version.

In [ ]:
from huggingface_hub import snapshot_download

try:
    snapshot_download(repo_id=HF_REPO, allow_patterns=["results/*"],
                      local_dir=".", repo_type="model")
    existing = sorted(Path("results").glob("*.jsonl"))
    print("recovered result files:", [p.name for p in existing] or "none")
    for p in existing:
        print(f"  {p.name}: {sum(1 for _ in open(p, encoding='utf-8'))} rows")
except Exception as exc:
    print("no existing results to resume from:", type(exc).__name__, exc)

## 5. Evaluation harness

One row per example, flushed and fsynced before the next example starts, so a kill loses at most the example in flight. Re-running skips ids already present. Every row keeps `raw_text`, `confidence` and `gold_answer` exactly as produced, with no thresholding.

In [ ]:
import gc, time

def load_done(path, retry_errors=False):
    """Ids already written. Tolerates a truncated last line from a hard kill."""
    if not Path(path).exists():
        return set()
    done, dropped = set(), 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                dropped += 1
                continue
            if retry_errors and row.get("error"):
                continue
            done.add(row.get("id"))
    if dropped:
        print(f"  note: ignored {dropped} unparsable line(s); they will be re-run")
    return done


def free_model():
    """Drop the cached model so the next config starts from a clean GPU."""
    from model.src import predict as P
    P._cache.update({"key": None, "model": None, "processor": None})
    gc.collect()
    torch.cuda.empty_cache()


def assert_generation_sane(res, config, split_name):
    """Abort the whole run if the first generation shows broken numerics.

    fp16 on a T4 yields NaN logits and empty text on this model. predict() now
    refuses to pick fp16, but a wrong dtype, a bad adapter merge or a driver
    problem would show the same way, and silently writing hundreds of empty
    rows would waste the whole commit. Checked once per configuration.
    """
    from model.src import predict as P
    d = P.LAST_GENERATION
    problems = []
    if not d["scores_finite"]:
        problems.append(
            f"logits were not finite ({d['non_finite_steps']} of "
            f"{d['new_tokens']} generated steps contained NaN or Inf)")
    if d["text_empty"]:
        problems.append("decoded output text was empty")
    if not d["new_tokens"]:
        problems.append("no tokens were generated")
    if problems:
        raise RuntimeError(
            "ABORTING: the first generation of config "
            f"'{config['name']}' on {split_name} is not usable.\n  "
            + "\n  ".join(problems)
            + f"\n\ncompute dtype : {d['dtype']}"
            + f"\ndevice capability: {d['device_capability']} "
              "(bf16 is native only from 8.0)"
            + f"\nraw_text        : {res.raw_text!r}"
            + f"\nconfidence      : {res.confidence}"
            + "\n\nfloat16 is a known-bad dtype for this model on a T4. "
              "predict.compute_dtype() should return bfloat16 on capability "
              ">= 8.0 and float32 otherwise, never float16."
        )
    print(f"  first-generation check OK: {d['new_tokens']} tokens, "
          f"finite logits, dtype={d['dtype']}, capability={d['device_capability']}")


def run_config(split_name, config, limit=None, retry_errors=False):
    if not cuda_available():
        raise RuntimeError("no usable GPU stack; refusing to produce stub rows")
    rows = SPLITS[split_name]
    out_path = Path("results") / f"{split_name}_{config['name']}.jsonl"
    out_path.parent.mkdir(parents=True, exist_ok=True)

    done = load_done(out_path, retry_errors)
    todo = [r for r in rows if r["id"] not in done]
    if limit:
        todo = todo[:limit]

    print(f"\n=== {split_name} / {config['name']} ===")
    print(f"  adapter={config['adapter'] or 'none'}  mode={config['mode']}  "
          f"prompt_style={config['prompt_style'] or 'default'}")
    print(f"  {len(done)} already done, {len(todo)} to do -> {out_path}")
    if not todo:
        return
    print("  example prompt:")
    for line in build_prompt(todo[0]["question"], config["mode"],
                             load_config()["abstain_answer"],
                             config["prompt_style"]).split("\n"):
        print("   |", line)

    t_start = time.time()
    n_ok = n_err = n_phrase = 0
    checked = False
    with open(out_path, "a", encoding="utf-8") as fh:
        for i, row in enumerate(todo, 1):
            rec = {
                "id": row["id"],
                "split": split_name,
                "config": config["name"],
                "mode": config["mode"],
                "prompt_style": config["prompt_style"] or "default",
                "adapter": config["adapter"] or None,
                "image_id": row.get("image_id"),
                "image_path": row["image_path"],
                "question": row["question"],
                "question_type": row.get("question_type"),
                "category": row.get("category"),
                "gold_answer": row["answer"],
                "threshold_applied": 0.0,
            }
            t0 = time.time()
            try:
                res = predict(DATA_ROOT / row["image_path"], row["question"],
                              mode=config["mode"],
                              adapter_path=config["adapter"],
                              prompt_style=config["prompt_style"])
                rec.update({
                    # raw_text is the pre-abstention text of record: predict()
                    # never redacts it, so thresholds can be applied offline.
                    "raw_text": res.raw_text,
                    "confidence": res.confidence,
                    "answer": res.answer,
                    # With threshold 0.0 this can only be True because the model
                    # itself emitted the abstain phrase - not a threshold effect.
                    "abstained_by_phrase": res.abstained,
                    "is_stub": res.is_stub,
                })
                n_ok += 1
                n_phrase += int(res.abstained)
                # Numeric sanity, once per config, before committing to a long run.
                if not checked:
                    assert_generation_sane(res, config, split_name)
                    checked = True
            except Exception as exc:
                rec["error"] = f"{type(exc).__name__}: {exc}"
                n_err += 1
                # A failed sanity check must stop the run, not become a data row.
                if isinstance(exc, RuntimeError) and str(exc).startswith("ABORTING:"):
                    raise
            rec["seconds"] = round(time.time() - t0, 2)

            # A stub row means the GPU stack has gone, which would be true for
            # every remaining example too. Abort before writing rather than
            # filling the file with hundreds of placeholder rows.
            if rec.get("is_stub"):
                raise RuntimeError(
                    "predict() returned a stub row: no usable GPU. Aborting so "
                    "the results file is not filled with placeholders."
                )

            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fh.flush()
            os.fsync(fh.fileno())

            if i % 25 == 0 or i == len(todo):
                rate = (time.time() - t_start) / i
                print(f"  [{i}/{len(todo)}] {rate:.1f}s/example  "
                      f"ok={n_ok} err={n_err} abstain_phrase={n_phrase}  "
                      f"eta {(len(todo)-i)*rate/60:.1f} min")

    print(f"  done in {(time.time()-t_start)/60:.1f} min "
          f"({n_ok} ok, {n_err} errors, {n_phrase} emitted the abstain phrase)")

## 6. Run the three configurations

`val.json` is processed before `test.json` within each configuration. The configuration loop is the outer one so the model loads three times rather than six — switching `adapter_path` forces a reload.

In [ ]:
CONFIGS = [
    {"name": "baseline",       "adapter": None,            "mode": "baseline",  "prompt_style": None},
    {"name": "adapter_train",  "adapter": str(ADAPTER),    "mode": "mitigated", "prompt_style": "train"},
    {"name": "adapter_strict", "adapter": str(ADAPTER),    "mode": "mitigated", "prompt_style": "strict"},
]

# Set to a small number for a fast end-to-end check before the full commit.
LIMIT = None

run_started = time.time()
for config in CONFIGS:
    free_model()
    for split_name in ("val", "test"):      # val always before test
        run_config(split_name, config, limit=LIMIT)
free_model()
print(f"\nALL CONFIGS DONE in {(time.time()-run_started)/60:.1f} min")

## 7. Sanity summary (no metrics, no thresholds)

In [ ]:
import collections

for path in sorted(Path("results").glob("*.jsonl")):
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    errs = [r for r in rows if r.get("error")]
    good = [r for r in rows if not r.get("error")]
    confs = [r["confidence"] for r in good]
    print(f"{path.name:<34} rows={len(rows):<5} errors={len(errs):<4} "
          f"stub={sum(r.get('is_stub', False) for r in good)}")
    if good:
        print(f"{'':34} mean confidence={sum(confs)/len(confs):.3f}  "
              f"abstain_phrase={sum(r['abstained_by_phrase'] for r in good)}")
        by_type = collections.Counter(r["question_type"] for r in good)
        print(f"{'':34} by question_type={dict(by_type)}")
    if errs:
        print(f"{'':34} first error: {errs[0]['error'][:90]}")

print("\nNo thresholds applied and no accuracy computed - that is deliberate.")
print("Sweep thresholds offline using the confidence + raw_text + gold_answer fields.")

## 8. Upload `results/` to the Hugging Face repo

In [ ]:
from huggingface_hub import HfApi

# The only step that needs write access, so it is the only one using the
# write token. Read lazily here and never printed.
HF_WRITE_TOKEN = UserSecretsClient().get_secret("HF_WRITE_TOKEN")

api = HfApi(token=HF_WRITE_TOKEN)
api.upload_folder(
    folder_path="results",
    path_in_repo="results",
    repo_id=HF_REPO,
    repo_type="model",
    commit_message="Add MedGemma eval results (baseline / adapter_train / adapter_strict)",
)
print("uploaded:")
for f in sorted(api.list_repo_files(HF_REPO, token=HF_WRITE_TOKEN)):
    if f.startswith("results/"):
        print("  ", f)

## Done

`results/<split>_<config>.jsonl` now holds one row per example for all three
configurations on both splits, and the same files are in the HF repo.

Each row carries `confidence`, `raw_text` and `gold_answer` as produced, with
`threshold_applied: 0.0`. Threshold selection, answer matching and metrics are
intentionally left to downstream analysis.